Look at snapshots of stratification over time

In [ ]:
import src.utils
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import seaborn as sns
import sklearn.linear_model
import pathlib
import os
import sklearn.linear_model
import cmocean

## RNG
rng = np.random.default_rng()

## color palette
sns.set(rc={"axes.facecolor": "white", "axes.grid": False}, palette="colorblind")

## paths
DATA_FP = pathlib.Path(os.environ["DATA_FP"])
# SAVE_FP = pathlib.Path(os.environ["SAVE_FP"])

Load data

In [ ]:
USE_WIDE = True

## load spatial data
forced, anom_ = src.utils.load_consolidated()

if USE_WIDE:

    ## load wide data
    forced_05, anom_05 = src.utils.load_consolidated_05()

    for v in list(forced_05):
        forced[v] = forced_05[v]
        anom_[v] = anom_05[v]

## get subset of data for total
VARNAMES = ["T"]
total = anom_[VARNAMES] + forced[VARNAMES]
total = xr.merge([forced[[f"{v}_comp" for v in VARNAMES]], total])
total = src.utils.get_windowed(total, stride=120)

In [ ]:
## compute clim
total_clim = total.sel(year=[1870, 2010, 2050, 2080]).mean(["time", "member"])
clim = src.utils.reconstruct_wrapper(total_clim, fn=lambda x: x)["T"]
clim_dTdz = clim.differentiate("z_t")

In [ ]:
AMP_DIFF = 2

## plot setup
fig, axs = plt.subplots(3, 3, figsize=(10, 7), layout="constrained")

for j, (Y0, Y1) in enumerate(zip([1870, 2010, 2050], [2010, 2050, 2080])):

    ## baseline
    for i, y in enumerate([Y0, Y1]):
        cp = axs[j, i].contourf(
            clim.longitude,
            clim.z_t,
            clim.sel(year=y),
            cmap="cmo.thermal",
            levels=np.arange(12, 34, 2),
            extend="both",
        )
        axs[j, i].set_title(y)

    ## difference
    cp_diff = axs[j, 2].contourf(
        clim.longitude,
        clim.z_t,
        clim.sel(year=Y1) - clim.sel(year=Y0),
        cmap="cmo.balance",
        levels=src.utils.make_cb_range(AMP_DIFF, AMP_DIFF / 10),
        extend="both",
    )
    axs[j, 2].set_title("Difference")

## set ax limit and plot Niño 3.4 bounds
for ax in axs.flatten():
    ax.set_xlim([140, 280])
    ax.set_ylim([150, 5])
    ax_kwargs = dict(ls="--", c="w", lw=0.8)
    ax.axvline(210, **ax_kwargs)
    ax.axvline(270, **ax_kwargs)
    ax.axhline(80, **ax_kwargs)

for ax in axs[:, 0]:
    ax.set_yticks([100, 0])
for ax in axs[:, 1:].flatten():
    ax.set_yticks([])
for ax in axs[:-1].flatten():
    ax.set_xticks([])
for ax in axs[-1]:
    ax.set_xticks([140, 210, 270])

plt.show()

Same, but stratification

In [ ]:
AMP_DIFF = 4e-2

## plot setup
fig, axs = plt.subplots(3, 3, figsize=(10, 7), layout="constrained")

for j, (Y0, Y1) in enumerate(zip([1870, 2010, 2050], [2010, 2050, 2080])):

    ## baseline
    for i, y in enumerate([Y0, Y1]):
        cp = axs[j, i].contourf(
            clim_dTdz.longitude,
            clim_dTdz.z_t,
            clim_dTdz.sel(year=y),
            cmap="cmo.amp_r",
            levels=np.arange(-2e-1, 2e-2, 2e-2),
            extend="both",
        )
        axs[j, i].set_title(y)

    ## difference
    cp_diff = axs[j, 2].contourf(
        clim_dTdz.longitude,
        clim_dTdz.z_t,
        clim_dTdz.sel(year=Y1) - clim_dTdz.sel(year=Y0),
        cmap="cmo.balance_r",
        levels=src.utils.make_cb_range(AMP_DIFF, AMP_DIFF / 10),
        extend="both",
    )
    axs[j, 2].set_title("Difference")

## set ax limit and plot Niño 3.4 bounds
for ax in axs.flatten():
    ax.set_xlim([140, 280])
    ax.set_ylim([150, 5])
    ax_kwargs = dict(ls="--", c="w", lw=0.8)
    ax.axvline(210, **ax_kwargs)
    ax.axvline(270, **ax_kwargs)
    ax.axhline(80, **ax_kwargs)

for ax in axs[:, 0]:
    ax.set_yticks([100, 0])
for ax in axs[:, 1:].flatten():
    ax.set_yticks([])
for ax in axs[:-1].flatten():
    ax.set_xticks([])
for ax in axs[-1]:
    ax.set_xticks([140, 210, 270])

plt.show()